
# Lesson 7: Parallel Execution in LangGraph

## 1. Learning Objectives

By the end of this lesson, we should understand:

- What parallel execution means in LangGraph.
- The difference between sequential and parallel execution.
- How to create parallel branches using graph edges.
- What fan-out and fan-in mean.
- How reducers combine updates from parallel nodes.
- What a super-step represents in LangGraph.
- How to synchronize parallel branches before executing a common node.

## 2. What Is Parallel Execution?

Parallel execution allows multiple independent nodes to execute concurrently instead of waiting for one another.

In sequential execution, the next node starts after the previous node finishes.

In parallel execution, multiple eligible nodes can execute during the same graph super-step.

Parallel execution is useful when tasks do not depend on each other's outputs.

Examples:
- Searching multiple sources simultaneously.
- Retrieving information from different databases.
- Running independent research tasks.
- Performing multiple independent data analyses.

## 3. Sequential vs Parallel Execution

### Sequential workflow

START → Python Research → LangGraph Research → END

Each node executes after the previous node completes.

### Parallel workflow

START → Python Research
START → LangGraph Research

Both research nodes can execute concurrently.

Their outputs are collected in the shared state.

## 4. Fan-out and Fan-in

### Fan-out

Fan-out means branching from one execution point into multiple independent tasks.

Example:
- Python research
- LangGraph research

### Fan-in

Fan-in means bringing multiple branches together into a common downstream node.

In our implementation, both research branches converge into a summary node.

The summary node executes after both branches complete.

## 5. Why Do We Need Reducers?

Parallel nodes may update the same state field.

For example:

Python Research returns:
`["Python is widely used in AI development."]`

LangGraph Research returns:
`["LangGraph helps build stateful AI workflows."]`

Both update the `results` field.

Without an appropriate reducer, LangGraph cannot combine multiple updates to the same state key in that super-step.

We therefore use:

`Annotated[list, operator.add]`

This tells LangGraph to concatenate the list updates.

## 6. Understanding Super-Steps

A super-step is a unit of graph execution in which one or more eligible nodes execute together.

In our workflow:

- Super-step 1: Python Research and LangGraph Research execute.
- Super-step 2: Summary executes after both research branches finish.

This synchronization ensures that the summary node receives both research results.

## 7. Practical Implementation: Parallel Research Agent

We implemented a research workflow with three nodes:

1. `python_research`: collects information about Python.
2. `langgraph_research`: collects information about LangGraph.
3. `summary`: combines the research findings into one summary.

### State Structure

The state contains:

- `results`: stores research findings and uses a reducer.
- `summary`: stores the final combined summary.

### Graph Architecture

START → Parallel Research Branches → Summary → END

### Parallel Branch Configuration

Both research nodes are connected to START:

```python
builder.add_edge(START, "python_research")
builder.add_edge(START, "langgraph_research")
```

This allows LangGraph to schedule both nodes in the same super-step.

### Fan-in Configuration

Both research nodes connect to the summary node:

```python
builder.add_edge("python_research", "summary")
builder.add_edge("langgraph_research", "summary")
```

LangGraph waits for both incoming branches to complete before executing the summary node.

## 8. Expected Output

The execution should show:

- Both research nodes executing.
- The summary node executing after both research nodes finish.
- Both research findings appearing in the final summary.

The order of parallel node execution and collected results is not guaranteed.

## 9. Key Concepts Summary

| Concept | Meaning |
|---|---|
| Parallel execution | Executing independent nodes concurrently |
| Fan-out | Branching into multiple tasks |
| Fan-in | Converging multiple branches |
| Super-step | A unit of graph execution |
| Reducer | Defines how multiple state updates are combined |
| Synchronization | Waiting for required branches before continuing |

## 10. Important Takeaways

1. Independent tasks can execute in parallel.
2. Parallel execution can reduce workflow waiting time.
3. Reducers are important when parallel nodes update the same state field.
4. Fan-in allows multiple branches to converge into a common node.
5. Node completion order should not be assumed in parallel workflows.

## 11. Interview Questions

**Q1. What is parallel execution in LangGraph?**

Executing multiple independent nodes concurrently within a graph workflow.

**Q2. What is fan-out?**

Splitting execution into multiple branches.

**Q3. What is fan-in?**

Converging multiple branches into a common downstream node.

**Q4. Why are reducers important in parallel execution?**

They define how updates from multiple nodes targeting the same state field are combined.

**Q5. What is a super-step?**

A unit of graph execution in which one or more eligible nodes execute together.

## 12. Next Lesson

Lesson 8: Cycles and Loops in LangGraph.

We will learn how to create iterative workflows where an agent can repeat tasks, evaluate results, and continue until a stopping condition is satisfied.


In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
import operator

In [7]:
class ResearchState(TypedDict):
    results: Annotated[list, operator.add]
    summary: str

In [6]:
def python_research(state: ResearchState):
    print("Researching Python...")

    return {
        "results": [
            "Python is widely used in AI development."
        ]
    }


def langgraph_research(state: ResearchState):
    print("Researching LangGraph...")

    return {
        "results": [
            "LangGraph helps build stateful AI workflows."
        ]
    }

def summary_node(state: ResearchState):
    print("Generating final summary...")

    summary = "Research findings:\n"

    for result in state["results"]:
        summary += f"- {result}\n"

    return {"summary": summary}

In [8]:
builder = StateGraph(ResearchState)

builder.add_node("python_research", python_research)
builder.add_node("langgraph_research", langgraph_research)
builder.add_node("summary", summary_node)

# Fan-out
builder.add_edge(START, "python_research")
builder.add_edge(START, "langgraph_research")

# Fan-in
builder.add_edge("python_research", "summary")
builder.add_edge("langgraph_research", "summary")

builder.add_edge("summary", END)

graph = builder.compile()

In [11]:
result = graph.invoke({
    "results": [],
    "summary": ""
})

print(result["summary"])

Researching LangGraph...
Researching Python...
Generating final summary...
Research findings:
- LangGraph helps build stateful AI workflows.
- Python is widely used in AI development.

